# Homework 4: Model Building and Tracking with MLflow

## Part 1. Load and Inspect the F1 Data

In this section, I load the F1 datasets from AWS/Databricks storage, clean the data types, and inspect missing values. Since the dataset contains missing values represented as `\N`, I treat them as nulls during loading. This step ensures the data is ready for feature engineering and model building in the next sections.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 1.1 Load the datasets

I use the F1 datasets provided for the course. The files are stored in the shared volume path. While loading the data, I convert `\N` values into nulls and cast important columns into appropriate numeric or date types.

In [0]:
# ---------- LOAD DATA ----------
base_path = "/Volumes/gr5069/raw/f1_data"

df_drivers = (
    spark.read
    .option("header", True)
    .option("nullValue", "\\N")
    .csv(f"{base_path}/drivers.csv")
    .withColumn("driverId", F.col("driverId").cast("int"))
    .withColumn("dob", F.to_date("dob", "yyyy-MM-dd"))
)

df_races = (
    spark.read
    .option("header", True)
    .option("nullValue", "\\N")
    .csv(f"{base_path}/races.csv")
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("round", F.col("round").cast("int"))
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))
)

df_results = (
    spark.read
    .option("header", True)
    .option("nullValue", "\\N")
    .csv(f"{base_path}/results.csv")
    .withColumn("resultId", F.col("resultId").cast("int"))
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("driverId", F.col("driverId").cast("int"))
    .withColumn("constructorId", F.col("constructorId").cast("int"))
    .withColumn("number", F.col("number").cast("int"))
    .withColumn("grid", F.col("grid").cast("int"))
    .withColumn("position", F.col("position").cast("int"))
    .withColumn("positionOrder", F.col("positionOrder").cast("int"))
    .withColumn("points", F.col("points").cast("double"))
    .withColumn("laps", F.col("laps").cast("int"))
    .withColumn("milliseconds", F.col("milliseconds").cast("bigint"))
    .withColumnRenamed("milliseconds", "race_milliseconds")
)

df_pitstops = (
    spark.read
    .option("header", True)
    .option("nullValue", "\\N")
    .csv(f"{base_path}/pit_stops.csv")
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("driverId", F.col("driverId").cast("int"))
    .withColumn("stop", F.col("stop").cast("int"))
    .withColumn("lap", F.col("lap").cast("int"))
    .withColumn("milliseconds", F.col("milliseconds").cast("bigint"))
    .withColumnRenamed("milliseconds", "pit_milliseconds")
)

## 1.2 Inspect the schema

After loading the data, I check the schema of each dataset to confirm that the columns were cast correctly. This is important because machine learning models require numeric features in the proper format.

In [0]:
print("drivers")
df_drivers.printSchema()

print("races")
df_races.printSchema()

print("results")
df_results.printSchema()

print("pit stops")
df_pitstops.printSchema()

## 1.3 Preview the datasets

I display a few rows from each table to verify that the records were loaded correctly and that the values look reasonable.

In [0]:
display(df_drivers.limit(5))
display(df_races.limit(5))
display(df_results.limit(5))
display(df_pitstops.limit(5))

## 1.4 Check missing values

Because this dataset contains missing values, I count the null values in each column. This helps identify which variables may need to be removed or imputed before building the model.

In [0]:
for name, df in [
    ("drivers", df_drivers),
    ("races", df_races),
    ("results", df_results),
    ("pit_stops", df_pitstops)
]:
    print(f"\n{name} null counts")
    display(
        df.select([
            F.count(F.when(F.col(c).isNull(), c)).alias(c)
            for c in df.columns
        ])
    )

## 1.5 Summary of this step

At this stage, the data has been successfully loaded and cleaned at a basic level. I converted the missing-value marker `\N` into nulls, casted key columns into numeric and date types, and inspected both schema and null counts. The null-value inspection shows that the main performance tables are usable for modeling. In the `results` table, the `position` column contains many missing values, which is expected because some drivers did not finish or were disqualified. However, columns such as `positionOrder`, `points`, `grid`, and `laps` are much more complete. The `pit_stops` table is relatively clean and can be aggregated into useful race-level driver features such as the number of pit stops and total pit stop time. The next step is to join the tables and create a modeling dataset for prediction.

## Part 2. Join Tables and Create the Modeling Dataset

In this section, I combine the race results table with race-level information and aggregated pit stop statistics. The goal is to create a clean modeling dataset for prediction. I use the `results` table as the main table, then join race features from `races` and pit stop summary features from `pit_stops`.

In [0]:
# ---------- AGGREGATE PIT STOP FEATURES ----------
df_pit_features = (
    df_pitstops
    .groupBy("raceId", "driverId")
    .agg(
        F.count("*").alias("pit_stop_count"),
        F.sum("pit_milliseconds").alias("total_pit_time"),
        F.avg("pit_milliseconds").alias("avg_pit_time")
    )
)

display(df_pit_features.limit(10))

### 2.1 Join the tables

I join the `results` table with selected features from the `races` table and the aggregated pit stop table. This creates one row per driver-race observation.

In [0]:
df_model = (
    df_results
    .join(
        df_races.select("raceId", "year", "round"),
        on="raceId",
        how="left"
    )
    .join(
        df_pit_features,
        on=["raceId", "driverId"],
        how="left"
    )
    .fillna({
        "pit_stop_count": 0,
        "total_pit_time": 0,
        "avg_pit_time": 0
    })
)

display(df_model.limit(10))
print("Row count:", df_model.count())

### 2.2 Select modeling variables

To keep the model simple and interpretable, I select a small set of numeric predictors and use `points` as the target variable. I also remove rows with missing values in key modeling columns.

In [0]:
model_cols = [
    "grid",
    "laps",
    "year",
    "round",
    "pit_stop_count",
    "total_pit_time",
    "avg_pit_time",
    "points"
]

df_model_final = (
    df_model
    .select(*model_cols)
    .dropna()
)

display(df_model_final.limit(10))
print("Final modeling row count:", df_model_final.count())

### 2.3 Final check of the modeling dataset

Before building the model, I inspect the final dataset and verify that the selected columns are complete and ready for machine learning.

In [0]:
df_model_final.printSchema()

display(
    df_model_final.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_model_final.columns
    ])
)

## Part 3. Build and Track the Model with MLflow

In this section, I build a machine learning model to predict race points using the modeling dataset created above. I use a Random Forest Regressor because it supports tunable hyperparameters and works well for structured tabular data. I track each run with MLflow by logging model parameters, evaluation metrics, the trained model itself, and artifacts such as prediction files and feature importance.

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
import tempfile

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### 3.1 Convert the Spark DataFrame to pandas

Since the modeling dataset is already cleaned and relatively compact, I convert it to a pandas DataFrame so that I can train a scikit-learn model and log the results with MLflow more easily.

In [0]:
pdf = df_model_final.toPandas()

print("Shape of modeling dataset:", pdf.shape)
display(pdf.head())

### 3.2 Define features and target

I use a small set of numeric predictors and define `points` as the regression target.

In [0]:
feature_cols = [
    "grid",
    "laps",
    "year",
    "round",
    "pit_stop_count",
    "total_pit_time",
    "avg_pit_time"
]

target_col = "points"

X = pdf[feature_cols]
y = pdf[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

### 3.3 Set up the MLflow experiment

I create or reuse an MLflow experiment for this homework so that all runs are stored in one place.

In [0]:
experiment_name = "/Users/wx2337@columbia.edu/f1_homework4_mlflow"

mlflow.set_experiment(experiment_name)
print("MLflow experiment set to:", experiment_name)

### 3.4 Run at least 10 experiments with different hyperparameters

To satisfy the assignment requirement, I train multiple Random Forest models using different parameter combinations. For each run, I log the hyperparameters, performance metrics, trained model, and at least two artifacts.

In [0]:
param_grid = {
    "n_estimators": [50, 100, 150, 200, 250],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5]
}

all_params = list(ParameterGrid(param_grid))
selected_params = all_params[:10]   # keep the first 10 combinations

print("Number of runs to execute:", len(selected_params))
selected_params

### 3.5 Train, evaluate, and log each run

For each experiment, I train the model, compute regression metrics, and log the outputs to MLflow. I also save artifacts including a prediction file, a feature-importance file, and a residual plot.

In [0]:
run_summaries = []

for i, params in enumerate(selected_params, start=1):
    with mlflow.start_run(run_name=f"rf_run_{i}"):
        # Train model
        model = RandomForestRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            random_state=42,
            n_jobs=-1
        )
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Metrics
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        
        # Log params
        mlflow.log_param("model_type", "RandomForestRegressor")
        mlflow.log_param("target", target_col)
        mlflow.log_param("features", ", ".join(feature_cols))
        mlflow.log_param("n_estimators", params["n_estimators"])
        mlflow.log_param("max_depth", params["max_depth"])
        mlflow.log_param("min_samples_split", params["min_samples_split"])
        mlflow.log_param("random_state", 42)
        
        # Log metrics
        mlflow.log_metric("mae", float(mae))
        mlflow.log_metric("mse", float(mse))
        mlflow.log_metric("rmse", float(rmse))
        mlflow.log_metric("r2", float(r2))
        
        # Log model
        mlflow.sklearn.log_model(model, "random_forest_model")
        
        # Temporary folder for artifacts
        temp_dir = tempfile.mkdtemp()
        
        try:
            # Artifact 1: predictions CSV
            pred_df = pd.DataFrame({
                "actual_points": y_test.values,
                "predicted_points": y_pred,
                "residual": y_test.values - y_pred
            })
            pred_path = os.path.join(temp_dir, "predictions.csv")
            pred_df.to_csv(pred_path, index=False)
            mlflow.log_artifact(pred_path)
            
            # Artifact 2: feature importance CSV
            fi_df = pd.DataFrame({
                "feature": feature_cols,
                "importance": model.feature_importances_
            }).sort_values("importance", ascending=False)
            fi_path = os.path.join(temp_dir, "feature_importance.csv")
            fi_df.to_csv(fi_path, index=False)
            mlflow.log_artifact(fi_path)
            
            # Artifact 3: residual plot
            plt.figure(figsize=(8, 5))
            plt.scatter(y_pred, y_test.values - y_pred, alpha=0.5)
            plt.axhline(y=0)
            plt.xlabel("Predicted Points")
            plt.ylabel("Residuals")
            plt.title(f"Residual Plot - Run {i}")
            plot_path = os.path.join(temp_dir, "residual_plot.png")
            plt.savefig(plot_path, bbox_inches="tight")
            plt.close()
            mlflow.log_artifact(plot_path)
        
        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)
        
        # Save summary for comparison
        run_summaries.append({
            "run_number": i,
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "min_samples_split": params["min_samples_split"],
            "mae": mae,
            "mse": mse,
            "rmse": rmse,
            "r2": r2
        })

print("Finished logging all runs to MLflow.")

### 3.6 Compare the runs

After logging all runs, I compare the experiment results in a summary table to identify the strongest model configuration.

In [0]:
results_df = pd.DataFrame(run_summaries).sort_values(by="rmse", ascending=True)
display(results_df)

### 3.7 Select the best run

I define the best model as the one with the lowest RMSE on the test set. I also review R² and MAE to confirm that the selected run performs well across multiple regression metrics.

In [0]:
best_run = results_df.iloc[0]
print("Best run based on lowest RMSE:")
print(best_run)

## Part 4. Interpretation of the Best Model

Based on the MLflow experiment results, I select the run with the lowest RMSE as the best model. This run provides the most accurate point predictions on the test set among the 10 parameter combinations tested. I also consider the R² value and MAE to confirm that the model performs consistently across multiple metrics. The selected Random Forest configuration offers a reasonable balance between predictive performance and model complexity.

In [0]:
best_run_dict = best_run.to_dict()

print("Best model summary")
for k, v in best_run_dict.items():
    print(f"{k}: {v}")